# A Case Study of Simulations with Context-Specific Salmonella GEMs

**Author:** Gina Vazquez  
**Date:** April 24, 2026 

## Goal

## Dataset

## Notes

## Workflow


In [31]:
import os
import pandas as pd

import imatpy
from imatpy.model_utils import model_eq

import cobra
from cobra.core import Configuration
from cobra.io import load_json_model
from cobra.flux_analysis import (
    flux_variability_analysis, pfba, geometric_fba, single_gene_deletion,
    single_reaction_deletion, double_gene_deletion,double_reaction_deletion)

In [32]:
os.environ["GRB_LICENSE_FILE"] = "/home/gmvaz/.gurobi.lic"
Configuration().solver = "gurobi"

# Run FBA on all models

In [4]:
# Load the input model of Salmonella Tm. - stm_v1_0
stm_v1 = load_json_model("STM_v1_0.json")

# Load the extracted Salmonella models. 
# Mixed - 278 sample of cs/bth/stm
# Stm - 279 sample of Salmonella
mixed1090 = load_json_model("run_031526/context_models/mixed1090_imat_restrictions.json")
mixed1585 = load_json_model("run_031526/context_models/mixed1585_imat_restrictions.json")
mixed2575 = load_json_model("run_031526/context_models/mixed2575_imat_restrictions.json")
stm1090 = load_json_model("run_031526/context_models/stm1090_imat_restrictions.json")
stm1585 = load_json_model("run_031526/context_models/stm1585_imat_restrictions.json")
stm2575 = load_json_model("run_031526/context_models/stm2575_imat_restrictions.json")

Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2789960
Academic license 2789960 - for non-commercial use only - registered to gm___@ucdavis.edu


In [34]:
models = {
    "input_stm_v1": stm_v1,
    "mixed_10_90": mixed1090,
    "mixed_15_85": mixed1585,
    "mixed_25_75": mixed2575,
    "stm_10_90": stm1090,
    "stm_15_85": stm1585,
    "stm_25_75": stm2575,
}

In [35]:
fba_results = []

for model_name, model in models.items():
    solution = model.optimize()

    fba_results.append({
        "model": model_name,
        "status": solution.status,
        "objective_value": solution.objective_value,
        "n_reactions": len(model.reactions),
        "n_metabolites": len(model.metabolites),
        "n_genes": len(model.genes),
    })

fba_df = pd.DataFrame(fba_results)

display(fba_df)

,model,status,objective_value,n_reactions,n_metabolites,n_genes
0,input_stm_v1,optimal,0.477834,2545,1802,1271
1,mixed_10_90,optimal,0.477834,2545,1802,1271
2,mixed_15_85,optimal,0.477834,2545,1802,1271
3,mixed_25_75,optimal,0.477834,2545,1802,1271
4,stm_10_90,optimal,0.477834,2545,1802,1271
5,stm_15_85,optimal,0.477834,2545,1802,1271
6,stm_25_75,optimal,0.477834,2545,1802,1271


In [36]:
input_growth = fba_df.loc[
    fba_df["model"] == "input_stm_v1",
    "objective_value"
].iloc[0]

fba_df["difference_vs_input"] = fba_df["objective_value"] - input_growth
fba_df["fraction_of_input"] = fba_df["objective_value"] / input_growth

display(fba_df.sort_values("fraction_of_input"))

,model,status,objective_value,n_reactions,n_metabolites,n_genes,difference_vs_input,fraction_of_input
0,input_stm_v1,optimal,0.477834,2545,1802,1271,0.000000e+00,1.0
2,mixed_15_85,optimal,0.477834,2545,1802,1271,0.000000e+00,1.0
3,mixed_25_75,optimal,0.477834,2545,1802,1271,0.000000e+00,1.0
4,stm_10_90,optimal,0.477834,2545,1802,1271,0.000000e+00,1.0
6,stm_25_75,optimal,0.477834,2545,1802,1271,0.000000e+00,1.0
5,stm_15_85,optimal,0.477834,2545,1802,1271,0.000000e+00,1.0
1,mixed_10_90,optimal,0.477834,2545,1802,1271,1.221245e-15,1.0


In [37]:
fba_df.to_csv("fba_objective_values_all_models.csv", index=False)

In [38]:
#fba_df.sort_values("fraction_of_input").style.background_gradient(
#    subset=["objective_value", "fraction_of_input"]
#)

ImportError: background_gradient requires matplotlib.

# Compare where the models differ

The input model and the extracted models only differ in reaction 3HAD180. I don't know what it means that they have different subsystems. 

In [5]:
# Check if the models are equal and where they differ
model_eq(stm_v1, mixed1090, True)
model_eq(stm_v1, mixed1585, True)
model_eq(stm_v1, mixed2575, True)
model_eq(stm_v1, stm1090, True)
model_eq(stm_v1, stm1585, True)
model_eq(stm_v1, stm2575, True)

Verbose model comparison
Reaction 3HAD180 has different subsystems
Verbose model comparison
Reaction 3HAD180 has different subsystems
Verbose model comparison
Reaction 3HAD180 has different subsystems
Verbose model comparison
Reaction 3HAD180 has different subsystems
Verbose model comparison
Reaction 3HAD180 has different subsystems
Verbose model comparison
Reaction 3HAD180 has different subsystems


False

There's a value error comparing any two extracted models to each other. I don't know what this means.  

In [6]:
# model_eq(mixed1090, stm2575)

# Run FBA

In [7]:
stmv1fba = stm_v1.optimize()
stm_v1.summary()

Metabolite,Reaction,Flux,C-Number,C-Flux
ca2_e,EX_ca2_e,0.002263,0,0.00%
cl_e,EX_cl_e,0.002263,0,0.00%
cobalt2_e,EX_cobalt2_e,0.001509,0,0.00%
cu2_e,EX_cu2_e,0.001509,0,0.00%
fe2_e,EX_fe2_e,0.003502,0,0.00%
glc__D_e,EX_glc__D_e,5,6,100.00%
k_e,EX_k_e,0.08486,0,0.00%
mg2_e,EX_mg2_e,0.003772,0,0.00%
mn2_e,EX_mn2_e,0.001509,0,0.00%
mobd_e,EX_mobd_e,0.001509,0,0.00%


In [8]:
mixed1090fba = mixed1090.optimize()
mixed1090.summary()

Metabolite,Reaction,Flux,C-Number,C-Flux
ca2_e,EX_ca2_e,0.002263,0,0.00%
cl_e,EX_cl_e,0.002263,0,0.00%
cobalt2_e,EX_cobalt2_e,0.001509,0,0.00%
cu2_e,EX_cu2_e,0.001509,0,0.00%
fe2_e,EX_fe2_e,0.003502,0,0.00%
glc__D_e,EX_glc__D_e,5,6,100.00%
k_e,EX_k_e,0.08486,0,0.00%
mg2_e,EX_mg2_e,0.003772,0,0.00%
mn2_e,EX_mn2_e,0.001509,0,0.00%
mobd_e,EX_mobd_e,0.001509,0,0.00%


# Run FVA

There is an option to run FVA loopless but there's an error sayin it's infeasible for the solver. 

In [9]:
flux_variability_analysis(stm_v1)

,minimum,maximum
3HAD180,0.002079,0.002079
3HAD181,0.003118,0.003118
3HAD40,0.105576,0.105576
3HAD60,0.105576,0.105576
3HAD80,0.105576,0.105576
...,...,...
12dgr2_ST,0.000045,0.000045
OAL_ST,0.000448,0.000448
OA4L_ST,0.000000,0.000000
OA5L_ST,0.000448,0.000448


In [10]:
stm_v1.summary(fva=0.90)

Metabolite,Reaction,Flux,Range,C-Number,C-Flux
ca2_e,EX_ca2_e,0.002263,[0.002037; 0.002263],0,0.00%
cl_e,EX_cl_e,0.002263,[0.002037; 0.002263],0,0.00%
cobalt2_e,EX_cobalt2_e,0.001509,[0.001358; 0.001509],0,0.00%
cu2_e,EX_cu2_e,0.001509,[0.001358; 0.001509],0,0.00%
fe2_e,EX_fe2_e,0.003502,[0; 33.87],0,0.00%
glc__D_e,EX_glc__D_e,5,[4.535; 5],6,100.00%
k_e,EX_k_e,0.08486,[0.07638; 0.08486],0,0.00%
mg2_e,EX_mg2_e,0.003772,[0.003395; 0.003772],0,0.00%
mn2_e,EX_mn2_e,0.001509,[0.001358; 0.001509],0,0.00%
mobd_e,EX_mobd_e,0.001509,[0.001358; 0.001509],0,0.00%


In [11]:
mixed1090.summary(fva=0.90)

Metabolite,Reaction,Flux,Range,C-Number,C-Flux
ca2_e,EX_ca2_e,0.002263,[0.002037; 0.002263],0,0.00%
cl_e,EX_cl_e,0.002263,[0.002037; 0.002263],0,0.00%
cobalt2_e,EX_cobalt2_e,0.001509,[0.001358; 0.001509],0,0.00%
cu2_e,EX_cu2_e,0.001509,[0.001358; 0.001509],0,0.00%
fe2_e,EX_fe2_e,0.003502,[0; 33.87],0,0.00%
glc__D_e,EX_glc__D_e,5,[4.535; 5],6,100.00%
k_e,EX_k_e,0.08486,[0.07638; 0.08486],0,0.00%
mg2_e,EX_mg2_e,0.003772,[0.003395; 0.003772],0,0.00%
mn2_e,EX_mn2_e,0.001509,[0.001358; 0.001509],0,0.00%
mobd_e,EX_mobd_e,0.001509,[0.001358; 0.001509],0,0.00%


In [12]:
stmv1_pfba = pfba(stm_v1)
print(stmv1_pfba)

<Solution 398.537 at 0x7cb3cb6bdc90>


# Simulate Deletions

In [13]:
# single gene deletions
stmv1_single_deletions = single_gene_deletion(stm_v1)

In [14]:
print(stmv1_single_deletions)

            ids    growth   status
0     {STM2041}  0.477834  optimal
1     {STM0221}  0.477834  optimal
2     {STM0588}  0.477834  optimal
3     {STM0774}  0.477834  optimal
4     {STM0568}  0.477834  optimal
...         ...       ...      ...
1266  {STM3720}  0.000000  optimal
1267  {STM3053}  0.477834  optimal
1268  {STM3722}  0.000000  optimal
1269  {STM3040}  0.477834  optimal
1270  {STM1208}  0.477834  optimal

[1271 rows x 3 columns]


In [15]:
mixed1090_single_deletions = single_gene_deletion(mixed1090)

In [16]:
print(mixed1090_single_deletions)

            ids    growth   status
0     {STM3686}  0.477834  optimal
1     {STM2976}  0.477834  optimal
2     {STM2326}  0.407010  optimal
3     {STM2017}  0.477834  optimal
4     {STM2190}  0.477834  optimal
...         ...       ...      ...
1266  {STM2074}  0.000000  optimal
1267  {STM2863}  0.477834  optimal
1268  {STM2531}  0.477834  optimal
1269  {STM4220}  0.477834  optimal
1270  {STM3924}  0.477834  optimal

[1271 rows x 3 columns]


In [27]:
keywords = ["sdh", "frd", "ldh", "lld", "dld", "cyo"]

for gene in stm_v1.genes:
    if any(k in gene.name.lower() for k in keywords):
        print(gene.id, gene.name)

STM4340 frdD
STM4342 frdB
STM0441 cyoC
STM0442 cyoB
STM4343 frdA
STM4341 frdC
STM0440 cyoD
STM0443 cyoA
STM3692 lldP
STM1647 ldhA
STM2167 dld
STM3694 lldD
STM0439 cyoE
STM0528 allD
STM0733 sdhD
STM0732 sdhC
STM0735 sdhB
STM0734 sdhA


In [28]:
print(mixed1090_single_deletions.knockout[
      "STM0733", "STM0732", "STM0735",
      "STM0734", "STM4340", "STM4342",
      "STM4343", "STM4341", "STM3692", 
      "STM1647", "STM2167", "STM3694",
      "STM0441", "STM0440", "STM0442",
      "STM0443", "STM0439"])

            ids    growth   status
58    {STM4343}  0.477834  optimal
150   {STM4340}  0.477834  optimal
193   {STM0439}  0.477834  optimal
260   {STM2167}  0.477834  optimal
298   {STM0442}  0.418156  optimal
335   {STM0735}  0.457016  optimal
364   {STM1647}  0.477834  optimal
415   {STM0732}  0.457016  optimal
608   {STM4341}  0.477834  optimal
613   {STM0440}  0.418156  optimal
617   {STM4342}  0.477834  optimal
698   {STM3694}  0.477834  optimal
706   {STM0443}  0.418156  optimal
1059  {STM3692}  0.477834  optimal
1066  {STM0733}  0.457016  optimal
1205  {STM0441}  0.418156  optimal
1218  {STM0734}  0.457016  optimal


In [ ]:
# Problematic simulations to run 

# double_gene_deletion(stm_v1, stm_v1.genes).round(4)
# double_gene_deletion(mixed1090, mixed1090.genes).round(4)

#mixed1090_pfba = pfba(mixed1090)
#print(mixed1090_pfba)

# stmv1_geomfba = geometric_fba(stm_v1)
# stmv1_geomfba

# mixed1090_pfba = geometric_fba(mixed1090)